# Generalization & Robustness

Two experiments testing how well the MIT-BIH-trained pipeline generalizes:
1. **Amplitude Invariance** — robustness to signal amplitude scaling (simulating different leads/hardware)
2. **Cross-Lead Validation** — transfer from MIT-BIH MLII to INCART 12-lead dataset

# Amplitude Invariance Experiment

**Problem:** Reducing R-amplitude (simulating pericardial effusion/cardiomyopathy) causes the classifier to flip V beats → S beats, because raw amplitude features lose discriminative power. Record 208 (~40% PVC burden) demonstrates this clearly in the dashboard.

**Root cause:** 5 raw amplitude-dependent features (`r_amplitude`, `qrs_max`, `qrs_min`, `qrs_range`, `qrs_area`) shift under amplitude scaling. The model still sees "premature" (timing) but no longer sees "ventricular origin" (morphology), so it classifies as S.

**Two options tested:**
1. **Pre-normalize amplitude** before feature extraction — scale each record to a reference amplitude so the classifier sees consistent morphology
2. **Drop raw amplitude features** — remove the 5 amplitude-dependent features (29 remaining), keeping only features that are already amplitude-invariant

**Evaluation:** Train on DS1, test on DS2 at original amplitude and at 50%, 30%, 20% amplitude reduction.

In [ ]:
import numpy as np
import pandas as pd
import wfdb
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, confusion_matrix

from ecg_monitor.pipeline import (
    FEATURE_COLS, DS1_RECORDS, DS2_RECORDS, DATA_DIR, LABEL_MAP,
    PACED_RECORDS, BEAT_SYMBOLS,
    bandpass_filter, build_df_all, get_train_test_split,
    extract_beat_features, pan_tompkins_detect,
)

# Amplitude-dependent features (5 total)
RAW_AMP_FEATURES = ['r_amplitude', 'qrs_max', 'qrs_min', 'qrs_range', 'qrs_area']

# Option 2: feature set with raw amplitude features removed
FEATURE_COLS_NO_AMP = [f for f in FEATURE_COLS if f not in RAW_AMP_FEATURES]
print(f"Full feature set: {len(FEATURE_COLS)} features")
print(f"No-amp feature set: {len(FEATURE_COLS_NO_AMP)} features")
print(f"Dropped: {RAW_AMP_FEATURES}")

In [ ]:
# Load dataset and cache raw signals for DS2 (needed for amplitude scaling)
df_all, pca = build_df_all(verbose=True)
df_train, df_test = get_train_test_split(df_all)

print(f"\nDS1 (train): {len(df_train)} beats")
print(f"DS2 (test):  {len(df_test)} beats")

# Cache raw signals + annotations for all records
raw_data = {}
all_records = open(f'{DATA_DIR}/RECORDS').read().strip().split('\n')
for rec_id in all_records:
    if rec_id in PACED_RECORDS:
        continue
    record = wfdb.rdrecord(f'{DATA_DIR}/{rec_id}')
    ann = wfdb.rdann(f'{DATA_DIR}/{rec_id}', 'atr')
    raw_data[rec_id] = {
        'signal': record.p_signal[:, 0],
        'fs': record.fs,
        'r_peaks': ann.sample,
        'symbols': list(ann.symbol),
    }

## Helper: Rebuild features with amplitude scaling + optional normalization

Replicates the inner loop of `build_df_all()` but inserts amplitude scaling (and optional normalization to a target amplitude) between filtering and feature extraction.

In [ ]:
from sklearn.decomposition import PCA

def compute_record_amplitude(filtered_signal, r_peaks):
    """Median absolute R-peak amplitude for a record."""
    valid = r_peaks[(r_peaks >= 0) & (r_peaks < len(filtered_signal))]
    if len(valid) == 0:
        return 1.0
    return float(np.median(np.abs(filtered_signal[valid])))


def build_features_scaled(record_ids, raw_data, scale_factor=1.0,
                          target_amp=None, pca_model=None,
                          pca_fit_ids=None, n_pca=10):
    """Build feature DataFrame with amplitude scaling and optional normalization.
    
    Args:
        record_ids: which records to process
        raw_data: dict of cached raw signals/annotations
        scale_factor: multiply raw signal by this before processing
        target_amp: if provided, normalize each record's amplitude to this value
                    (applied AFTER scale_factor, before feature extraction)
        pca_model: pre-fitted PCA to use. If None, fits a new one on pca_fit_ids.
        pca_fit_ids: records to fit PCA on (only used if pca_model is None)
        n_pca: number of PCA components
    
    Returns:
        df: feature DataFrame with clinical labels
        pca_model: the PCA model used (fitted or passed in)
    """
    all_features = []
    
    for rec_id in record_ids:
        if rec_id not in raw_data:
            continue
        rd = raw_data[rec_id]
        signal = rd['signal'] * scale_factor
        fs = rd['fs']
        
        _, filtered = pan_tompkins_detect(signal, fs)
        
        # Optional: normalize to target amplitude
        if target_amp is not None:
            rec_amp = compute_record_amplitude(filtered, rd['r_peaks'])
            if rec_amp > 0:
                filtered = filtered * (target_amp / rec_amp)
        
        df_rec = extract_beat_features(filtered, fs, rd['r_peaks'], rd['symbols'])
        df_rec['record'] = rec_id
        all_features.append(df_rec)
    
    df = pd.concat(all_features, ignore_index=True)
    
    # Label mapping
    df['clinical_label'] = df['label'].map(LABEL_MAP)
    df = df[df['clinical_label'].notna()].copy()
    
    # Per-patient normalized morphology
    for feat in ['r_amplitude', 'qrs_width_ms', 'qrs_area']:
        group_mean = df.groupby('record')[feat].transform('mean')
        group_std = df.groupby('record')[feat].transform('std').replace(0, 1)
        df[f'{feat}_norm'] = (df[feat] - group_mean) / group_std
    
    # PCA on beat waveforms
    waveform_matrix = np.stack(df['beat_waveform'].values)
    if pca_model is None:
        pca_model = PCA(n_components=n_pca)
        if pca_fit_ids:
            fit_mask = df['record'].isin(pca_fit_ids)
            pca_model.fit(waveform_matrix[fit_mask])
        else:
            pca_model.fit(waveform_matrix)
    
    pca_features = pca_model.transform(waveform_matrix)
    for i in range(pca_model.n_components):
        df[f'pca_{i}'] = pca_features[:, i]
    df = df.drop(columns=['beat_waveform'])
    
    return df, pca_model


def train_gb(df_train, feature_cols):
    """Train HistGradientBoosting with standard hyperparameters."""
    X = df_train[feature_cols].values
    y = df_train['clinical_label'].values
    sw = compute_sample_weight('balanced', y)
    clf = HistGradientBoostingClassifier(
        max_iter=300, max_depth=6, learning_rate=0.1,
        min_samples_leaf=20, l2_regularization=1.0, random_state=42,
    )
    clf.fit(X, y, sample_weight=sw)
    return clf


def eval_model(clf, df_test, feature_cols, label=''):
    """Evaluate and print classification report. Returns per-class F1 dict."""
    X = df_test[feature_cols].values
    y_true = df_test['clinical_label'].values
    y_pred = clf.predict(X)
    
    if label:
        print(f"\n{'='*60}")
        print(f"  {label}")
        print(f"{'='*60}")
    print(classification_report(y_true, y_pred, digits=3))
    
    report = classification_report(y_true, y_pred, output_dict=True)
    return {
        'accuracy': report['accuracy'],
        'N_f1': report.get('N', {}).get('f1-score', 0),
        'S_f1': report.get('S', {}).get('f1-score', 0),
        'V_f1': report.get('V', {}).get('f1-score', 0),
        'macro_f1': report['macro avg']['f1-score'],
    }


def eval_record(clf, df_test, feature_cols, record_id):
    """Evaluate on a single record, return metrics dict."""
    df_rec = df_test[df_test['record'] == record_id]
    if len(df_rec) == 0:
        return {}
    y_true = df_rec['clinical_label'].values
    y_pred = clf.predict(df_rec[feature_cols].values)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return {
        'accuracy': report['accuracy'],
        'N_f1': report.get('N', {}).get('f1-score', 0),
        'S_f1': report.get('S', {}).get('f1-score', 0),
        'V_f1': report.get('V', {}).get('f1-score', 0),
        'macro_f1': report['macro avg']['f1-score'],
        'V_pred': int(np.sum(y_pred == 'V')),
        'S_pred': int(np.sum(y_pred == 'S')),
        'V_true': int(np.sum(y_true == 'V')),
        'S_true': int(np.sum(y_true == 'S')),
    }

print("Helpers defined.")

## Baseline: 34-feature GB on DS2 (original amplitude)

In [ ]:
# Train baseline model on original data
clf_baseline = train_gb(df_train, FEATURE_COLS)
baseline_results = eval_model(clf_baseline, df_test, FEATURE_COLS, "Baseline (34 features, original amplitude)")

# Record 208 specifically (DS1, but let's check a high-PVC DS2 record too)
# Record 208 is DS1 — find DS2 records with high PVC burden for robustness test
for rid in sorted(DS2_RECORDS):
    df_rec = df_test[df_test['record'] == rid]
    n_v = (df_rec['clinical_label'] == 'V').sum()
    n_s = (df_rec['clinical_label'] == 'S').sum()
    pvc_pct = 100 * n_v / len(df_rec) if len(df_rec) > 0 else 0
    sve_pct = 100 * n_s / len(df_rec) if len(df_rec) > 0 else 0
    if pvc_pct > 5 or sve_pct > 5:
        print(f"Record {rid}: {len(df_rec)} beats, PVC {pvc_pct:.1f}%, SVE {sve_pct:.1f}%")

## Option 2: Drop raw amplitude features (29 features)

Simpler option first — remove the 5 amplitude-dependent features and check accuracy on unscaled data.

In [ ]:
clf_no_amp = train_gb(df_train, FEATURE_COLS_NO_AMP)
no_amp_results = eval_model(clf_no_amp, df_test, FEATURE_COLS_NO_AMP, "Option 2: No raw amplitude features (29 features)")

## Option 1: Pre-normalize amplitude before feature extraction

Compute a target amplitude from DS1, then normalize every record's filtered signal to that target before extracting features. This makes raw amplitude features encode morphology (shape) rather than signal gain.

In [ ]:
# Compute target amplitude from DS1 records (median of per-record median R-peak amplitudes)
ds1_amps = []
for rec_id in sorted(DS1_RECORDS):
    rd = raw_data[rec_id]
    _, filtered = pan_tompkins_detect(rd['signal'], rd['fs'])
    rec_amp = compute_record_amplitude(filtered, rd['r_peaks'])
    ds1_amps.append(rec_amp)
    print(f"Record {rec_id}: median |R-peak| = {rec_amp:.4f} mV")

target_amp = float(np.median(ds1_amps))
print(f"\nTarget amplitude (median of DS1): {target_amp:.4f} mV")

In [ ]:
# Build amplitude-normalized features for all records
all_record_ids = sorted(set(list(DS1_RECORDS) + list(DS2_RECORDS)))

df_normed, pca_normed = build_features_scaled(
    all_record_ids, raw_data,
    scale_factor=1.0, target_amp=target_amp,
    pca_model=None, pca_fit_ids=DS1_RECORDS,
)

# Split into train/test
train_mask = df_normed['record'].isin(DS1_RECORDS)
test_mask = df_normed['record'].isin(DS2_RECORDS)
df_train_normed = df_normed[train_mask].copy()
df_test_normed = df_normed[test_mask].copy()

print(f"Amp-normalized DS1: {len(df_train_normed)} beats")
print(f"Amp-normalized DS2: {len(df_test_normed)} beats")

# Train and evaluate
clf_ampnorm = train_gb(df_train_normed, FEATURE_COLS)
ampnorm_results = eval_model(clf_ampnorm, df_test_normed, FEATURE_COLS,
                              "Option 1: Amplitude pre-normalized (34 features)")

## Baseline comparison at original amplitude

Quick sanity check: how do the three approaches compare on unscaled DS2?

In [ ]:
comparison = pd.DataFrame([
    {'Model': 'Baseline (34 feat)', **baseline_results},
    {'Model': 'Option 1: Amp-normalized (34 feat)', **ampnorm_results},
    {'Model': 'Option 2: No raw amp (29 feat)', **no_amp_results},
]).set_index('Model')

comparison.columns = ['Accuracy', 'N F1', 'S F1', 'V F1', 'Macro F1']
print(comparison.round(3).to_string())

## Robustness test: amplitude reduction on DS2

For each approach, scale DS2 signals to 50%, 30%, 20% of original amplitude, rebuild features, and classify with the pre-trained models. This simulates the amplitude reduction transformation from the dashboard.

In [ ]:
scale_factors = [1.0, 0.5, 0.3, 0.2]
results_rows = []

for sf in scale_factors:
    print(f"\n{'#'*60}")
    print(f"  Scale factor: {sf}")
    print(f"{'#'*60}")
    
    # --- Baseline: rebuild DS2 features at scaled amplitude, classify with baseline model ---
    df_ds2_scaled, _ = build_features_scaled(
        sorted(DS2_RECORDS), raw_data,
        scale_factor=sf, target_amp=None,
        pca_model=pca,  # use original PCA
    )
    r = eval_model(clf_baseline, df_ds2_scaled, FEATURE_COLS,
                   f"Baseline @ {int(sf*100)}% amplitude")
    results_rows.append({'model': 'Baseline (34)', 'scale': sf, **r})
    
    # --- Option 1: rebuild DS2 at scaled amplitude WITH amp normalization ---
    df_ds2_normed, _ = build_features_scaled(
        sorted(DS2_RECORDS), raw_data,
        scale_factor=sf, target_amp=target_amp,
        pca_model=pca_normed,  # use amp-normalized PCA
    )
    r = eval_model(clf_ampnorm, df_ds2_normed, FEATURE_COLS,
                   f"Option 1 (amp-norm) @ {int(sf*100)}% amplitude")
    results_rows.append({'model': 'Opt1: Amp-norm (34)', 'scale': sf, **r})
    
    # --- Option 2: rebuild DS2 at scaled amplitude, classify with no-amp model ---
    # (reuse df_ds2_scaled from baseline — same features, just use fewer columns)
    r = eval_model(clf_no_amp, df_ds2_scaled, FEATURE_COLS_NO_AMP,
                   f"Option 2 (no raw amp) @ {int(sf*100)}% amplitude")
    results_rows.append({'model': 'Opt2: No raw amp (29)', 'scale': sf, **r})

df_results = pd.DataFrame(results_rows)
print("\nDone.")

## Summary table and degradation curves

In [ ]:
# Summary table
pivot = df_results.pivot_table(
    index='model',
    columns='scale',
    values=['macro_f1', 'V_f1', 'S_f1', 'accuracy'],
)
print("=== Macro F1 by scale factor ===")
print(df_results.pivot(index='model', columns='scale', values='macro_f1').round(3).to_string())
print("\n=== V F1 by scale factor ===")
print(df_results.pivot(index='model', columns='scale', values='V_f1').round(3).to_string())
print("\n=== S F1 by scale factor ===")
print(df_results.pivot(index='model', columns='scale', values='S_f1').round(3).to_string())
print("\n=== Accuracy by scale factor ===")
print(df_results.pivot(index='model', columns='scale', values='accuracy').round(3).to_string())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics = [('macro_f1', 'Macro F1'), ('V_f1', 'V-class F1'), ('S_f1', 'S-class F1')]
colors = {'Baseline (34)': '#e74c3c', 'Opt1: Amp-norm (34)': '#2ecc71', 'Opt2: No raw amp (29)': '#3498db'}
markers = {'Baseline (34)': 'o', 'Opt1: Amp-norm (34)': 's', 'Opt2: No raw amp (29)': '^'}

for ax, (metric, title) in zip(axes, metrics):
    for model_name in df_results['model'].unique():
        subset = df_results[df_results['model'] == model_name].sort_values('scale', ascending=False)
        ax.plot(subset['scale'] * 100, subset[metric],
                marker=markers.get(model_name, 'o'),
                color=colors.get(model_name, 'gray'),
                label=model_name, linewidth=2, markersize=8)
    ax.set_xlabel('Amplitude (%)')
    ax.set_ylabel(title)
    ax.set_title(title)
    ax.set_xlim(15, 105)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.invert_xaxis()

plt.suptitle('Classification Robustness Under Amplitude Reduction', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Record-level deep dive: high-PVC DS2 records under amplitude reduction

Check V→S flip behavior on DS2 records with high PVC burden.

In [ ]:
# Identify DS2 records with V beats for the deep dive
pvc_records = []
for rid in sorted(DS2_RECORDS):
    df_rec = df_test[df_test['record'] == rid]
    n_v = (df_rec['clinical_label'] == 'V').sum()
    if n_v > 10:  # at least 10 V beats
        pvc_records.append(rid)

print(f"DS2 records with >10 V beats: {pvc_records}")

# For each scale factor and each approach, evaluate per-record on high-PVC records
record_rows = []

for sf in scale_factors:
    # Baseline scaled
    df_ds2_scaled, _ = build_features_scaled(
        sorted(DS2_RECORDS), raw_data,
        scale_factor=sf, target_amp=None, pca_model=pca,
    )
    # Option 1 scaled + normalized
    df_ds2_normed, _ = build_features_scaled(
        sorted(DS2_RECORDS), raw_data,
        scale_factor=sf, target_amp=target_amp, pca_model=pca_normed,
    )
    
    for rid in pvc_records:
        # Baseline
        r = eval_record(clf_baseline, df_ds2_scaled, FEATURE_COLS, rid)
        record_rows.append({'record': rid, 'model': 'Baseline', 'scale': sf, **r})
        
        # Option 1
        r = eval_record(clf_ampnorm, df_ds2_normed, FEATURE_COLS, rid)
        record_rows.append({'record': rid, 'model': 'Opt1: Amp-norm', 'scale': sf, **r})
        
        # Option 2
        r = eval_record(clf_no_amp, df_ds2_scaled, FEATURE_COLS_NO_AMP, rid)
        record_rows.append({'record': rid, 'model': 'Opt2: No raw amp', 'scale': sf, **r})

df_records = pd.DataFrame(record_rows)
print("Done.")

In [ ]:
# Show V F1 degradation per record per model
for rid in pvc_records:
    df_r = df_records[df_records['record'] == rid]
    n_v_true = df_r['V_true'].iloc[0] if 'V_true' in df_r.columns else '?'
    print(f"\n--- Record {rid} (V beats: {n_v_true}) ---")
    pivot = df_r.pivot(index='model', columns='scale', values='V_f1')
    print(pivot.round(3).to_string())
    
    # Show V→S flip: how many S predictions at each scale?
    print("\n  V predicted / S predicted:")
    for _, row in df_r.iterrows():
        print(f"  {row['model']:20s} @ {row['scale']:.0%}: V_pred={row.get('V_pred','?'):>4}, S_pred={row.get('S_pred','?'):>4}")

## Confusion matrices at 20% amplitude (worst case)

Show the V→S flip pattern explicitly for each approach at the most extreme reduction.

In [ ]:
# Rebuild DS2 at 20% for confusion matrices
df_ds2_20, _ = build_features_scaled(
    sorted(DS2_RECORDS), raw_data,
    scale_factor=0.2, target_amp=None, pca_model=pca,
)
df_ds2_20_normed, _ = build_features_scaled(
    sorted(DS2_RECORDS), raw_data,
    scale_factor=0.2, target_amp=target_amp, pca_model=pca_normed,
)

labels_order = ['N', 'S', 'V']
models_20 = [
    ("Baseline @ 20%", clf_baseline, df_ds2_20, FEATURE_COLS),
    ("Opt1: Amp-norm @ 20%", clf_ampnorm, df_ds2_20_normed, FEATURE_COLS),
    ("Opt2: No raw amp @ 20%", clf_no_amp, df_ds2_20, FEATURE_COLS_NO_AMP),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (title, clf, df_eval, feat_cols) in zip(axes, models_20):
    y_true = df_eval['clinical_label'].values
    y_pred = clf.predict(df_eval[feat_cols].values)
    cm = confusion_matrix(y_true, y_pred, labels=labels_order)
    
    # Normalize by row (true label)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cm[i,j]}\n({cm_norm[i,j]:.0%})",
                    ha='center', va='center', fontsize=9,
                    color='white' if cm_norm[i,j] > 0.5 else 'black')
    ax.set_xticks(range(3))
    ax.set_xticklabels(labels_order)
    ax.set_yticks(range(3))
    ax.set_yticklabels(labels_order)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title, fontsize=10)

plt.suptitle('Confusion Matrices at 20% Amplitude', fontsize=13)
plt.tight_layout()
plt.show()

## Conclusions

In [ ]:
# Final summary: degradation from 100% to 20% amplitude
print("=== Macro F1 degradation (100% → 20% amplitude) ===\n")
for model_name in df_results['model'].unique():
    subset = df_results[df_results['model'] == model_name]
    f1_100 = subset[subset['scale'] == 1.0]['macro_f1'].values[0]
    f1_20 = subset[subset['scale'] == 0.2]['macro_f1'].values[0]
    drop = f1_100 - f1_20
    print(f"{model_name:30s}  100%: {f1_100:.3f}  20%: {f1_20:.3f}  drop: {drop:+.3f}")

print("\n=== V F1 degradation (100% → 20% amplitude) ===\n")
for model_name in df_results['model'].unique():
    subset = df_results[df_results['model'] == model_name]
    f1_100 = subset[subset['scale'] == 1.0]['V_f1'].values[0]
    f1_20 = subset[subset['scale'] == 0.2]['V_f1'].values[0]
    drop = f1_100 - f1_20
    print(f"{model_name:30s}  100%: {f1_100:.3f}  20%: {f1_20:.3f}  drop: {drop:+.3f}")

## Hybrid 59-feature model (34 base + 25 P-wave)

Repeat the amplitude invariance experiment with the full hybrid model from Results_Summary.ipynb. This includes adaptive P-wave raw features, per-patient-normalized P-wave features, fixed + adaptive template correlation, and P-wave PCA.

P-wave features include raw amplitude-dependent ones (`pw2_max`, `pw2_min`, `pw2_range`, `pw2_peak_amplitude`, `pw2_area`, `pw2_energy`) — these will also shift under amplitude reduction. The per-patient-normalized versions (`_pnorm`) and template correlations should be invariant.

In [ ]:
# --- P-wave feature extraction functions (from Results_Summary.ipynb) ---

from scipy.signal import find_peaks, resample as sig_resample

PW_RESAMPLE_LEN = 44  # fixed resampled length (~120ms at 360 Hz)
N_PW_PCA = 5


def extract_pwave_features_v1(filtered, fs, df_rec):
    """Extract P-wave features from FIXED window (200ms to 80ms before R-peak)."""
    pw_start_offset = int(0.200 * fs)
    pw_end_offset = int(0.080 * fs)
    pw_len = pw_start_offset - pw_end_offset
    sample_indices = df_rec['sample_idx'].values

    nan_row = {k: np.nan for k in [
        'pw_peak_amplitude', 'pw_energy', 'pw_peak_prominence',
        'pw_has_peak', 'pw_baseline_dev', 'pw_max', 'pw_min', 'pw_range', 'pw_area']}

    features = []
    for _, row in df_rec.iterrows():
        r_curr = int(row['sample_idx'])
        pw_start = r_curr - pw_start_offset
        pw_end = r_curr - pw_end_offset

        pos = np.searchsorted(sample_indices, r_curr)
        if pos > 0:
            prev_qrs_end = int(sample_indices[pos - 1]) + int(0.100 * fs)
            if pw_start < prev_qrs_end:
                pw_start = prev_qrs_end

        if pw_start < 0 or pw_end >= len(filtered) or pw_end <= pw_start or (pw_end - pw_start) < 5:
            feat = nan_row.copy()
            feat['sample_idx'] = r_curr
            feat['pw_waveform'] = None
            features.append(feat)
            continue

        pw_segment = filtered[pw_start:pw_end]
        baseline = (np.mean(pw_segment[:3]) + np.mean(pw_segment[-3:])) / 2
        pw_detrended = pw_segment - baseline

        peaks, props = find_peaks(pw_detrended, prominence=0.01)
        has_peak = len(peaks) > 0
        if has_peak:
            best = np.argmax(props['prominences'])
            peak_amp = pw_detrended[peaks[best]]
            peak_prom = props['prominences'][best]
        else:
            peak_amp = np.max(pw_detrended)
            peak_prom = 0.0

        pw_resampled = sig_resample(pw_segment, pw_len) if len(pw_segment) >= 5 else None

        features.append({
            'sample_idx': r_curr, 'pw_waveform': pw_resampled,
            'pw_peak_amplitude': peak_amp, 'pw_energy': np.sum(pw_detrended**2) / len(pw_detrended),
            'pw_peak_prominence': peak_prom, 'pw_has_peak': float(has_peak),
            'pw_baseline_dev': np.mean(np.abs(pw_detrended)),
            'pw_max': np.max(pw_segment), 'pw_min': np.min(pw_segment),
            'pw_range': np.max(pw_segment) - np.min(pw_segment),
            'pw_area': np.trapezoid(pw_detrended) / fs,
        })
    return pd.DataFrame(features)


def extract_pwave_features_v2(filtered, fs, df_rec):
    """Extract P-wave features from ADAPTIVE window (25% to 10% of preceding RR)."""
    sample_indices = df_rec['sample_idx'].values

    nan_row = {k: np.nan for k in [
        'pw2_peak_amplitude', 'pw2_energy', 'pw2_peak_prominence',
        'pw2_has_peak', 'pw2_baseline_dev', 'pw2_max', 'pw2_min', 'pw2_range', 'pw2_area']}

    features = []
    for _, row in df_rec.iterrows():
        r_curr = int(row['sample_idx'])
        pos = np.searchsorted(sample_indices, r_curr)
        rr_samples = (r_curr - int(sample_indices[pos - 1])) if pos > 0 else int(0.8 * fs)

        pw_start = r_curr - int(0.25 * rr_samples)
        pw_end = r_curr - int(0.10 * rr_samples)

        if pos > 0:
            prev_qrs_end = int(sample_indices[pos - 1]) + int(0.100 * fs)
            if pw_start < prev_qrs_end:
                pw_start = prev_qrs_end

        min_window = int(0.015 * fs)
        if pw_start < 0 or pw_end >= len(filtered) or pw_end <= pw_start or (pw_end - pw_start) < min_window:
            feat = nan_row.copy()
            feat['sample_idx'] = r_curr
            feat['pw2_waveform'] = None
            features.append(feat)
            continue

        pw_segment = filtered[pw_start:pw_end]
        n_edge = max(1, min(3, len(pw_segment) // 3))
        baseline = (np.mean(pw_segment[:n_edge]) + np.mean(pw_segment[-n_edge:])) / 2
        pw_detrended = pw_segment - baseline

        peaks, props = find_peaks(pw_detrended, prominence=0.01)
        has_peak = len(peaks) > 0
        if has_peak:
            best = np.argmax(props['prominences'])
            peak_amp = pw_detrended[peaks[best]]
            peak_prom = props['prominences'][best]
        else:
            peak_amp = np.max(pw_detrended)
            peak_prom = 0.0

        pw_resampled = sig_resample(pw_segment, PW_RESAMPLE_LEN)

        features.append({
            'sample_idx': r_curr, 'pw2_waveform': pw_resampled,
            'pw2_peak_amplitude': peak_amp, 'pw2_energy': np.sum(pw_detrended**2) / len(pw_detrended),
            'pw2_peak_prominence': peak_prom, 'pw2_has_peak': float(has_peak),
            'pw2_baseline_dev': np.mean(np.abs(pw_detrended)),
            'pw2_max': np.max(pw_segment), 'pw2_min': np.min(pw_segment),
            'pw2_range': np.max(pw_segment) - np.min(pw_segment),
            'pw2_area': np.trapezoid(pw_detrended) / fs,
        })
    return pd.DataFrame(features)


PW2_FEAT_COLS = ['pw2_peak_amplitude', 'pw2_energy', 'pw2_peak_prominence',
                 'pw2_has_peak', 'pw2_baseline_dev', 'pw2_max', 'pw2_min',
                 'pw2_range', 'pw2_area']
PW2_PNORM_COLS = [f'{feat}_pnorm' for feat in PW2_FEAT_COLS]
pw_pca_cols = [f'pw_pca_{i}' for i in range(N_PW_PCA)]

print("P-wave extraction functions defined.")

In [ ]:
def build_hybrid_features(record_ids, raw_data, scale_factor=1.0,
                          target_amp=None, qrs_pca_model=None,
                          qrs_pca_fit_ids=None, pw_pca_model=None):
    """Build full 59-feature hybrid DataFrame with optional amplitude scaling/normalization.
    
    Steps:
    1. Scale raw signal, filter, optionally normalize amplitude
    2. Extract base 34 features (beat features + per-patient norm + QRS PCA)
    3. Extract P-wave v1 features (fixed window) for template correlation
    4. Extract P-wave v2 features (adaptive window) for raw/pnorm/PCA
    5. Merge into hybrid dataset with template correlation + PCA + per-patient norm
    
    Returns: (df_hybrid, qrs_pca_model, pw_pca_model)
    """
    all_base = []
    all_pw1 = []
    all_pw2 = []
    
    for rec_id in record_ids:
        if rec_id not in raw_data:
            continue
        rd = raw_data[rec_id]
        signal = rd['signal'] * scale_factor
        fs = rd['fs']
        
        _, filtered = pan_tompkins_detect(signal, fs)
        
        if target_amp is not None:
            rec_amp = compute_record_amplitude(filtered, rd['r_peaks'])
            if rec_amp > 0:
                filtered = filtered * (target_amp / rec_amp)
        
        # Base features
        df_rec = extract_beat_features(filtered, fs, rd['r_peaks'], rd['symbols'])
        df_rec['record'] = rec_id
        all_base.append(df_rec)
        
        # P-wave v1 (fixed window) — for template correlation
        pw1 = extract_pwave_features_v1(filtered, fs, df_rec)
        all_pw1.append(pw1)
        
        # P-wave v2 (adaptive window) — for raw features + PCA
        pw2 = extract_pwave_features_v2(filtered, fs, df_rec)
        all_pw2.append(pw2)
    
    df_base = pd.concat(all_base, ignore_index=True)
    df_pw1 = pd.concat(all_pw1, ignore_index=True)
    df_pw2 = pd.concat(all_pw2, ignore_index=True)
    
    # --- Label mapping ---
    df_base['clinical_label'] = df_base['label'].map(LABEL_MAP)
    df_base = df_base[df_base['clinical_label'].notna()].copy()
    
    # Align P-wave DataFrames (same rows as df_base after label filtering)
    valid_idx = df_base.index
    df_pw1 = df_pw1.loc[valid_idx].reset_index(drop=True)
    df_pw2 = df_pw2.loc[valid_idx].reset_index(drop=True)
    df_base = df_base.reset_index(drop=True)
    
    # --- Per-patient normalization (base morphology) ---
    for feat in ['r_amplitude', 'qrs_width_ms', 'qrs_area']:
        group_mean = df_base.groupby('record')[feat].transform('mean')
        group_std = df_base.groupby('record')[feat].transform('std').replace(0, 1)
        df_base[f'{feat}_norm'] = (df_base[feat] - group_mean) / group_std
    
    # --- QRS PCA ---
    waveform_matrix = np.stack(df_base['beat_waveform'].values)
    if qrs_pca_model is None:
        qrs_pca_model = PCA(n_components=10)
        if qrs_pca_fit_ids:
            fit_mask = df_base['record'].isin(qrs_pca_fit_ids)
            qrs_pca_model.fit(waveform_matrix[fit_mask])
        else:
            qrs_pca_model.fit(waveform_matrix)
    pca_features = qrs_pca_model.transform(waveform_matrix)
    for i in range(qrs_pca_model.n_components):
        df_base[f'pca_{i}'] = pca_features[:, i]
    df_base = df_base.drop(columns=['beat_waveform'])
    
    # --- Merge P-wave v2 raw features ---
    for col in PW2_FEAT_COLS:
        df_base[col] = df_pw2[col].values
    df_base['pw2_waveform'] = df_pw2['pw2_waveform'].values
    
    # Drop rows with NaN P-wave features
    valid_mask = df_base[PW2_FEAT_COLS].notna().all(axis=1)
    df_base = df_base[valid_mask].copy()
    df_pw1 = df_pw1.loc[valid_mask.values].reset_index(drop=True)
    df_base = df_base.reset_index(drop=True)
    
    # --- P-wave v1 template correlation (fixed window, per-patient) ---
    pw1_waveforms = df_pw1['pw_waveform'].values
    pw_len_v1 = int(0.200 * 360) - int(0.080 * 360)
    
    df_base['pw_template_corr'] = 0.0
    for rec in df_base['record'].unique():
        rec_mask = df_base['record'] == rec
        n_mask = rec_mask & (df_base['clinical_label'] == 'N')
        
        n_waveforms = []
        for wf in pw1_waveforms[n_mask.values]:
            if wf is not None and hasattr(wf, '__len__') and len(wf) == pw_len_v1:
                n_waveforms.append(wf)
        
        if len(n_waveforms) < 10:
            continue
        
        template = np.mean(n_waveforms, axis=0)
        template_c = template - np.mean(template)
        template_std = np.std(template_c)
        if template_std == 0:
            continue
        
        indices = df_base.index[rec_mask]
        for idx in indices:
            wf = pw1_waveforms[idx]
            if wf is not None and hasattr(wf, '__len__') and len(wf) == pw_len_v1:
                beat_c = wf - np.mean(wf)
                beat_std = np.std(beat_c)
                if beat_std > 0:
                    df_base.loc[idx, 'pw_template_corr'] = np.corrcoef(template_c, beat_c)[0, 1]
    
    # --- P-wave v2 template correlation (adaptive window, per-patient) ---
    df_base['pw2_template_corr'] = 0.0
    for rec in df_base['record'].unique():
        rec_mask = df_base['record'] == rec
        n_mask = rec_mask & (df_base['clinical_label'] == 'N')
        
        n_waveforms = []
        for wf in df_base.loc[n_mask, 'pw2_waveform']:
            if wf is not None and hasattr(wf, '__len__'):
                if len(wf) == PW_RESAMPLE_LEN:
                    n_waveforms.append(wf)
                elif len(wf) >= 5:
                    n_waveforms.append(sig_resample(wf, PW_RESAMPLE_LEN))
        
        if len(n_waveforms) < 10:
            continue
        
        template = np.mean(n_waveforms, axis=0)
        template_c = template - np.mean(template)
        template_std = np.std(template_c)
        if template_std == 0:
            continue
        
        for idx in df_base.index[rec_mask]:
            wf = df_base.loc[idx, 'pw2_waveform']
            if wf is not None and hasattr(wf, '__len__') and len(wf) >= 5:
                if len(wf) != PW_RESAMPLE_LEN:
                    wf = sig_resample(wf, PW_RESAMPLE_LEN)
                beat_c = wf - np.mean(wf)
                beat_std = np.std(beat_c)
                if beat_std > 0:
                    df_base.loc[idx, 'pw2_template_corr'] = np.corrcoef(template_c, beat_c)[0, 1]
    
    # --- P-wave PCA (adaptive window) ---
    all_wf = []
    valid_pca_idx = []
    for idx in df_base.index:
        wf = df_base.loc[idx, 'pw2_waveform']
        if wf is not None and hasattr(wf, '__len__'):
            if len(wf) == PW_RESAMPLE_LEN:
                all_wf.append(wf)
                valid_pca_idx.append(idx)
            elif len(wf) >= 5:
                all_wf.append(sig_resample(wf, PW_RESAMPLE_LEN))
                valid_pca_idx.append(idx)
    
    all_wf_matrix = np.array(all_wf)
    
    if pw_pca_model is None:
        pw_pca_model = PCA(n_components=N_PW_PCA, random_state=42)
        # Fit on DS1 N beats
        ds1_n_idx = [i for i, idx in enumerate(valid_pca_idx)
                     if df_base.loc[idx, 'record'] in DS1_RECORDS
                     and df_base.loc[idx, 'clinical_label'] == 'N']
        if len(ds1_n_idx) > 0:
            pw_pca_model.fit(all_wf_matrix[ds1_n_idx])
    
    pca_transformed = pw_pca_model.transform(all_wf_matrix)
    for i, col in enumerate(pw_pca_cols):
        df_base[col] = 0.0
        for j, idx in enumerate(valid_pca_idx):
            df_base.loc[idx, col] = pca_transformed[j, i]
    
    # --- Per-patient normalization of P-wave v2 features (N-beat z-score) ---
    for feat in PW2_FEAT_COLS:
        n_stats = df_base[df_base['clinical_label'] == 'N'].groupby('record')[feat].agg(['mean', 'std'])
        n_stats['std'] = n_stats['std'].replace(0, 1)
        
        df_base[f'{feat}_pnorm'] = np.nan
        for rec in df_base['record'].unique():
            if rec in n_stats.index:
                rec_mask = df_base['record'] == rec
                df_base.loc[rec_mask, f'{feat}_pnorm'] = (
                    (df_base.loc[rec_mask, feat] - n_stats.loc[rec, 'mean']) / n_stats.loc[rec, 'std'])
    
    # Fill NaN pnorm/PCA with 0
    for col in PW2_PNORM_COLS + pw_pca_cols:
        df_base[col] = df_base[col].fillna(0.0)
    
    # Drop pw2_waveform column
    df_base = df_base.drop(columns=['pw2_waveform'], errors='ignore')
    
    # Drop rows still having NaN in pnorm columns
    df_base = df_base.dropna(subset=PW2_PNORM_COLS).copy()
    
    return df_base, qrs_pca_model, pw_pca_model


# Hybrid feature set (59 features)
FEATURES_HYBRID = (FEATURE_COLS + PW2_FEAT_COLS + PW2_PNORM_COLS +
                   ['pw_template_corr', 'pw2_template_corr'] + pw_pca_cols)

# Hybrid without raw amplitude features (from base AND P-wave)
RAW_AMP_PW = ['pw2_peak_amplitude', 'pw2_energy', 'pw2_max', 'pw2_min', 'pw2_range', 'pw2_area']
ALL_RAW_AMP = RAW_AMP_FEATURES + RAW_AMP_PW
FEATURES_HYBRID_NO_AMP = [f for f in FEATURES_HYBRID if f not in ALL_RAW_AMP]

print(f"Hybrid feature set: {len(FEATURES_HYBRID)} features")
print(f"Hybrid no-amp feature set: {len(FEATURES_HYBRID_NO_AMP)} features")
print(f"Dropped from hybrid: {ALL_RAW_AMP}")

In [ ]:
# Build hybrid features at original amplitude (all records)
print("Building hybrid features for all records (this takes a few minutes)...")
all_record_ids = sorted(set(list(DS1_RECORDS) + list(DS2_RECORDS)))

df_hybrid, qrs_pca_h, pw_pca_h = build_hybrid_features(
    all_record_ids, raw_data,
    scale_factor=1.0, target_amp=None,
    qrs_pca_fit_ids=DS1_RECORDS,
)

train_mask_h = df_hybrid['record'].isin(DS1_RECORDS)
df_train_h = df_hybrid[train_mask_h].copy()
df_test_h = df_hybrid[~train_mask_h].copy()

print(f"Hybrid train: {len(df_train_h)}, test: {len(df_test_h)}")
print(f"Feature columns present: {len([c for c in FEATURES_HYBRID if c in df_hybrid.columns])}/{len(FEATURES_HYBRID)}")

# Train hybrid baseline (59 features)
clf_hybrid = train_gb(df_train_h, FEATURES_HYBRID)
hybrid_results = eval_model(clf_hybrid, df_test_h, FEATURES_HYBRID, "Hybrid baseline (59 features)")

# Train hybrid no-amp
clf_hybrid_no_amp = train_gb(df_train_h, FEATURES_HYBRID_NO_AMP)
hybrid_no_amp_results = eval_model(clf_hybrid_no_amp, df_test_h, FEATURES_HYBRID_NO_AMP,
                                    f"Hybrid no raw amp ({len(FEATURES_HYBRID_NO_AMP)} features)")

### Hybrid robustness test: amplitude reduction

In [ ]:
hybrid_rows = []

for sf in scale_factors:
    print(f"\n--- Scale factor: {sf} ---")
    
    # Build DS2 hybrid features at scaled amplitude
    df_ds2_h_scaled, _, _ = build_hybrid_features(
        sorted(DS2_RECORDS), raw_data,
        scale_factor=sf, target_amp=None,
        qrs_pca_model=qrs_pca_h, pw_pca_model=pw_pca_h,
    )
    
    # Hybrid baseline (59 features)
    r = eval_model(clf_hybrid, df_ds2_h_scaled, FEATURES_HYBRID,
                   f"Hybrid (59) @ {int(sf*100)}%")
    hybrid_rows.append({'model': 'Hybrid (59)', 'scale': sf, **r})
    
    # Hybrid no-amp
    r = eval_model(clf_hybrid_no_amp, df_ds2_h_scaled, FEATURES_HYBRID_NO_AMP,
                   f"Hybrid no-amp ({len(FEATURES_HYBRID_NO_AMP)}) @ {int(sf*100)}%")
    hybrid_rows.append({'model': f'Hybrid no-amp ({len(FEATURES_HYBRID_NO_AMP)})', 'scale': sf, **r})

df_hybrid_results = pd.DataFrame(hybrid_rows)
print("\nDone.")

### Hybrid summary: all models compared

In [ ]:
# Combine all results into one table
df_all_results = pd.concat([df_results, df_hybrid_results], ignore_index=True)

print("=== Macro F1 by scale factor (all models) ===")
print(df_all_results.pivot(index='model', columns='scale', values='macro_f1').round(3).to_string())

print("\n=== V F1 by scale factor (all models) ===")
print(df_all_results.pivot(index='model', columns='scale', values='V_f1').round(3).to_string())

print("\n=== S F1 by scale factor (all models) ===")
print(df_all_results.pivot(index='model', columns='scale', values='S_f1').round(3).to_string())

In [ ]:
# Degradation curves — all 5 models
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics_plot = [('macro_f1', 'Macro F1'), ('V_f1', 'V-class F1'), ('S_f1', 'S-class F1')]

colors_all = {
    'Baseline (34)': '#e74c3c',
    'Opt1: Amp-norm (34)': '#95a5a6',
    'Opt2: No raw amp (29)': '#3498db',
    'Hybrid (59)': '#e67e22',
    f'Hybrid no-amp ({len(FEATURES_HYBRID_NO_AMP)})': '#2ecc71',
}
markers_all = {
    'Baseline (34)': 'o',
    'Opt1: Amp-norm (34)': 'D',
    'Opt2: No raw amp (29)': '^',
    'Hybrid (59)': 's',
    f'Hybrid no-amp ({len(FEATURES_HYBRID_NO_AMP)})': 'v',
}

for ax, (metric, title) in zip(axes, metrics_plot):
    for model_name in df_all_results['model'].unique():
        subset = df_all_results[df_all_results['model'] == model_name].sort_values('scale', ascending=False)
        ax.plot(subset['scale'] * 100, subset[metric],
                marker=markers_all.get(model_name, 'x'),
                color=colors_all.get(model_name, 'gray'),
                label=model_name, linewidth=2, markersize=8)
    ax.set_xlabel('Amplitude (%)')
    ax.set_ylabel(title)
    ax.set_title(title)
    ax.set_xlim(15, 105)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=7, loc='lower left')
    ax.grid(True, alpha=0.3)
    ax.invert_xaxis()

plt.suptitle('Classification Robustness Under Amplitude Reduction (All Models)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Final degradation summary — all models
print("=== Macro F1 degradation (100% → 20% amplitude) — All Models ===\n")
for model_name in df_all_results['model'].unique():
    subset = df_all_results[df_all_results['model'] == model_name]
    f1_100 = subset[subset['scale'] == 1.0]['macro_f1'].values[0]
    f1_20 = subset[subset['scale'] == 0.2]['macro_f1'].values[0]
    drop = f1_100 - f1_20
    print(f"{model_name:35s}  100%: {f1_100:.3f}  20%: {f1_20:.3f}  drop: {drop:+.3f}")

print("\n=== V F1 degradation (100% → 20% amplitude) — All Models ===\n")
for model_name in df_all_results['model'].unique():
    subset = df_all_results[df_all_results['model'] == model_name]
    f1_100 = subset[subset['scale'] == 1.0]['V_f1'].values[0]
    f1_20 = subset[subset['scale'] == 0.2]['V_f1'].values[0]
    drop = f1_100 - f1_20
    print(f"{model_name:35s}  100%: {f1_100:.3f}  20%: {f1_20:.3f}  drop: {drop:+.3f}")

print("\n=== S F1 degradation (100% → 20% amplitude) — All Models ===\n")
for model_name in df_all_results['model'].unique():
    subset = df_all_results[df_all_results['model'] == model_name]
    f1_100 = subset[subset['scale'] == 1.0]['S_f1'].values[0]
    f1_20 = subset[subset['scale'] == 0.2]['S_f1'].values[0]
    drop = f1_100 - f1_20
    print(f"{model_name:35s}  100%: {f1_100:.3f}  20%: {f1_20:.3f}  drop: {drop:+.3f}")

### Excluding record 232

In [ ]:
# Evaluate all models excluding record 232
df_test_no232 = df_test[df_test['record'] != '232']
df_test_h_no232 = df_test_h[df_test_h['record'] != '232']

models_excl232 = [
    ("Baseline (34)", clf_baseline, df_test_no232, FEATURE_COLS),
    ("Opt2: No raw amp (29)", clf_no_amp, df_test_no232, FEATURE_COLS_NO_AMP),
    ("Hybrid (59)", clf_hybrid, df_test_h_no232, FEATURES_HYBRID),
    ("Hybrid no-amp (48)", clf_hybrid_no_amp, df_test_h_no232, FEATURES_HYBRID_NO_AMP),
]

rows_232 = []
for name, clf, df_eval, feat_cols in models_excl232:
    r = eval_model(clf, df_eval, feat_cols, f"{name} — excluding record 232")
    rows_232.append({'Model': name, **r})

# Also show record 232 alone for context
df_test_232 = df_test[df_test['record'] == '232']
df_test_h_232 = df_test_h[df_test_h['record'] == '232']

models_232_only = [
    ("Baseline (34)", clf_baseline, df_test_232, FEATURE_COLS),
    ("Opt2: No raw amp (29)", clf_no_amp, df_test_232, FEATURE_COLS_NO_AMP),
    ("Hybrid (59)", clf_hybrid, df_test_h_232, FEATURES_HYBRID),
    ("Hybrid no-amp (48)", clf_hybrid_no_amp, df_test_h_232, FEATURES_HYBRID_NO_AMP),
]

rows_232_only = []
for name, clf, df_eval, feat_cols in models_232_only:
    r = eval_model(clf, df_eval, feat_cols, f"{name} — record 232 only")
    rows_232_only.append({'Model': name, **r})

print("\n=== Excluding record 232 ===")
df_excl = pd.DataFrame(rows_232).set_index('Model')
df_excl.columns = ['Accuracy', 'N F1', 'S F1', 'V F1', 'Macro F1']
print(df_excl.round(3).to_string())

print("\n=== Record 232 only ===")
df_only = pd.DataFrame(rows_232_only).set_index('Model')
df_only.columns = ['Accuracy', 'N F1', 'S F1', 'V F1', 'Macro F1']
print(df_only.round(3).to_string())

---

# Part 2: Cross-Lead Validation

In [ ]:
import wfdb
import numpy as np
import pandas as pd
from collections import Counter
from scipy.signal import butter, filtfilt, find_peaks
from scipy.stats import kurtosis, skew
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_sample_weight
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from ecg_monitor.pipeline import (
    build_df_all, get_train_test_split, bandpass_filter, pan_tompkins_detect,
    extract_beat_features, FEATURE_COLS, FEATURE_COLS_28, RR_ONLY_FEATURES,
    LABEL_MAP, BEAT_SYMBOLS, DS1_RECORDS, DS2_RECORDS, PACED_RECORDS
)

print("Imports OK")

## Train GB on MIT-BIH (DS1) — baseline model

Train the same HistGradientBoosting classifier on MIT-BIH DS1 with 34 features. Evaluate on DS2 to confirm baseline metrics match previous results.

In [ ]:
# Build MIT-BIH features and train GB classifier
print("Building MIT-BIH features...")
df_all, pca_mitbih = build_df_all()
df_train, df_test = get_train_test_split(df_all)

print(f"\nDS1 train: {len(df_train)} beats")
print(f"DS2 test:  {len(df_test)} beats")
print(f"\nDS1 class distribution:")
print(df_train['clinical_label'].value_counts().to_string())
print(f"\nDS2 class distribution:")
print(df_test['clinical_label'].value_counts().to_string())

# Train GB with 34 features (same hyperparameters as best model)
X_train = df_train[FEATURE_COLS].values
y_train = df_train['clinical_label'].values
sw_train = compute_sample_weight('balanced', y_train)

clf = HistGradientBoostingClassifier(
    max_iter=300, max_depth=6, learning_rate=0.1,
    min_samples_leaf=20, l2_regularization=1.0, random_state=42
)
clf.fit(X_train, y_train, sample_weight=sw_train)

# Baseline: evaluate on MIT-BIH DS2
X_test = df_test[FEATURE_COLS].values
y_test = df_test['clinical_label'].values
y_pred = clf.predict(X_test)

print("\n=== MIT-BIH DS2 Baseline (same-lead, same-fs) ===")
print(classification_report(y_test, y_pred, digits=3))
baseline_macro_f1 = f1_score(y_test, y_pred, average='macro')
baseline_per_class = {c: f1_score(y_test == c, y_pred == c) for c in ['N', 'S', 'V']}
print(f"Macro F1: {baseline_macro_f1:.3f}")

## Load INCART and extract features per lead

Download INCART records from PhysioNet. For each record, extract features from **Lead II** (closest to MLII) and **Lead I** (wearable-relevant). Use ground truth R-peaks from annotations — this isolates feature extraction and classification errors from R-peak detection errors.

Key differences from MIT-BIH:
- **257 Hz** vs 360 Hz → beat waveform window is 103 samples vs 144 samples → PCA needs retraining
- **Lead axis** differs → amplitude, shape features will shift; RR features unchanged

In [ ]:
# INCART: 75 records, 12-lead, 257 Hz, beat-level annotations
# Leads: ['I', 'II', 'III', 'AVR', 'AVL', 'AVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
# We test on Lead II (index 1, closest to MLII) and Lead I (index 0, wearable-relevant)

import os

INCART_FS = 257
INCART_LEADS = {'II': 1, 'I': 0, 'V1': 6, 'V2': 7}
TARGET_LEADS = ['II', 'I', 'V1', 'V2']
INCART_LOCAL = 'incartdb'  # local directory (pre-downloaded)

def load_incart_features(lead_name, lead_idx):
    """Load all INCART records from local disk, extract features from a single lead."""
    all_features = []
    loaded = 0
    
    for i in range(1, 76):
        rec_id = f'I{i:02d}'
        local_path = f'{INCART_LOCAL}/{rec_id}'
        
        # Try local first, fallback to PhysioNet
        try:
            if os.path.exists(f'{local_path}.hea'):
                record = wfdb.rdrecord(local_path)
                ann = wfdb.rdann(local_path, 'atr')
            else:
                record = wfdb.rdrecord(rec_id, pn_dir='incartdb')
                ann = wfdb.rdann(rec_id, 'atr', pn_dir='incartdb')
        except Exception as e:
            continue
            
        # Filter to beat annotations only
        beat_mask = [s in BEAT_SYMBOLS for s in ann.symbol]
        r_peaks = ann.sample[beat_mask]
        labels = [s for s, m in zip(ann.symbol, beat_mask) if m]
        
        if len(r_peaks) < 3:
            continue
        
        # Get single lead signal and filter
        raw_signal = record.p_signal[:, lead_idx]
        filtered = bandpass_filter(raw_signal, INCART_FS)
        
        # Extract features using the same pipeline (different fs)
        df_rec = extract_beat_features(filtered, INCART_FS, np.array(r_peaks), labels)
        if len(df_rec) == 0:
            continue
        df_rec['record'] = rec_id
        all_features.append(df_rec)
        loaded += 1
    
    df = pd.concat(all_features, ignore_index=True)
    
    # Label mapping (same as MIT-BIH)
    df['clinical_label'] = df['label'].map(LABEL_MAP)
    df = df[df['clinical_label'].notna()].copy()
    
    # Per-patient normalized morphology
    for feat in ['r_amplitude', 'qrs_width_ms', 'qrs_area']:
        group_mean = df.groupby('record')[feat].transform('mean')
        group_std = df.groupby('record')[feat].transform('std').replace(0, 1)
        df[f'{feat}_norm'] = (df[feat] - group_mean) / group_std
    
    # qrs_range_norm — needed for FEATURE_COLS compatibility
    group_mean = df.groupby('record')['qrs_range'].transform('mean')
    group_std = df.groupby('record')['qrs_range'].transform('std').replace(0, 1)
    df['qrs_range_norm'] = (df['qrs_range'] - group_mean) / group_std
    
    # PCA on beat waveforms — refit on INCART data (different fs = different waveform length)
    waveform_matrix = np.stack(df['beat_waveform'].values)
    pca_incart = PCA(n_components=10)
    pca_incart.fit(waveform_matrix)
    pca_features = pca_incart.transform(waveform_matrix)
    for j in range(10):
        df[f'pca_{j}'] = pca_features[:, j]
    df = df.drop(columns=['beat_waveform'])
    
    print(f"  Loaded {loaded} records")
    return df

# Load features for target leads
incart_dfs = {}
for lead_name in TARGET_LEADS:
    lead_idx = INCART_LEADS[lead_name]
    print(f"\nLoading INCART Lead {lead_name} (index {lead_idx})...")
    incart_dfs[lead_name] = load_incart_features(lead_name, lead_idx)
    df_lead = incart_dfs[lead_name]
    print(f"  Total beats: {len(df_lead)}")
    print(f"  Class distribution: {df_lead['clinical_label'].value_counts().to_dict()}")

## Cross-lead evaluation — MIT-BIH model on INCART leads

Apply the MIT-BIH-trained GB classifier directly to INCART features. Compare:
1. **Full 34 features** — includes lead-dependent morphology + PCA
2. **RR-only 13 features** — completely lead-independent (should transfer perfectly)

This reveals the cost of lead mismatch on morphology features.

In [ ]:
# Train RR-only model for comparison
clf_rr = HistGradientBoostingClassifier(
    max_iter=300, max_depth=6, learning_rate=0.1,
    min_samples_leaf=20, l2_regularization=1.0, random_state=42
)
clf_rr.fit(df_train[RR_ONLY_FEATURES].values, y_train, sample_weight=sw_train)

# MIT-BIH baselines first
y_pred_mitbih_full = clf.predict(X_test)
y_pred_mitbih_rr = clf_rr.predict(df_test[RR_ONLY_FEATURES].values)

results = []
results.append({
    'Lead': 'MIT-BIH MLII',
    'Model': 'Full (34)',
    'Accuracy': (y_pred_mitbih_full == y_test).mean(),
    'N F1': baseline_per_class['N'],
    'S F1': baseline_per_class['S'],
    'V F1': baseline_per_class['V'],
    'Macro F1': baseline_macro_f1,
})
results.append({
    'Lead': 'MIT-BIH MLII',
    'Model': 'RR-only (13)',
    'Accuracy': (y_pred_mitbih_rr == y_test).mean(),
    'N F1': f1_score(y_test == 'N', y_pred_mitbih_rr == 'N'),
    'S F1': f1_score(y_test == 'S', y_pred_mitbih_rr == 'S'),
    'V F1': f1_score(y_test == 'V', y_pred_mitbih_rr == 'V'),
    'Macro F1': f1_score(y_test, y_pred_mitbih_rr, average='macro', zero_division=0),
})

# Evaluate both models on each INCART lead
for lead_name in TARGET_LEADS:
    df_lead = incart_dfs[lead_name]
    y_true = df_lead['clinical_label'].values
    
    # Full 34-feature model
    X_full = df_lead[FEATURE_COLS].values
    y_pred_full = clf.predict(X_full)
    
    # RR-only 13-feature model
    X_rr = df_lead[RR_ONLY_FEATURES].values
    y_pred_rr = clf_rr.predict(X_rr)
    
    for model_name, y_p in [('Full (34)', y_pred_full), ('RR-only (13)', y_pred_rr)]:
        macro_f1 = f1_score(y_true, y_p, average='macro', zero_division=0)
        per_class_f1 = {}
        for c in ['N', 'S', 'V']:
            if (y_true == c).sum() > 0:
                per_class_f1[c] = f1_score(y_true == c, y_p == c)
            else:
                per_class_f1[c] = np.nan
        
        results.append({
            'Lead': lead_name,
            'Model': model_name,
            'Accuracy': (y_p == y_true).mean(),
            'N F1': per_class_f1['N'],
            'S F1': per_class_f1['S'],
            'V F1': per_class_f1['V'],
            'Macro F1': macro_f1,
        })

df_results = pd.DataFrame(results)
print("=== Cross-Lead Results: MIT-BIH model → INCART leads ===\n")
print(df_results.to_string(index=False, float_format='{:.3f}'.format))

## Feature distribution shift analysis

Compare feature distributions between MIT-BIH (MLII) and INCART (Lead II, Lead I). For each feature, compute the distribution shift (difference in mean, normalized by MIT-BIH std). This reveals which features are most affected by lead change vs sampling rate change.

In [ ]:
# Feature distribution shift: MIT-BIH vs INCART per lead
# Compute standardized mean difference (Cohen's d between datasets) for each feature

shift_rows = []
mitbih_means = df_all[FEATURE_COLS].mean()
mitbih_stds = df_all[FEATURE_COLS].std().replace(0, 1)

for lead_name in TARGET_LEADS:
    df_lead = incart_dfs[lead_name]
    for feat in FEATURE_COLS:
        incart_mean = df_lead[feat].mean()
        incart_std = df_lead[feat].std()
        # Standardized shift relative to MIT-BIH
        shift = abs(incart_mean - mitbih_means[feat]) / mitbih_stds[feat]
        shift_rows.append({
            'Feature': feat,
            'Lead': lead_name,
            'MIT-BIH mean': mitbih_means[feat],
            'INCART mean': incart_mean,
            'Shift (|d|)': shift,
        })

df_shift = pd.DataFrame(shift_rows)

# Show top shifted features for Lead II and Lead I
for lead_name in ['II', 'I']:
    sub = df_shift[df_shift['Lead'] == lead_name].sort_values('Shift (|d|)', ascending=False)
    print(f"\n=== Feature shift: MIT-BIH MLII → INCART Lead {lead_name} (top 15) ===")
    print(sub[['Feature', 'MIT-BIH mean', 'INCART mean', 'Shift (|d|)']].head(15).to_string(index=False, float_format='{:.3f}'.format))

# Categorize features by transferability
rr_features = set(RR_ONLY_FEATURES)
morphology_features = {'r_amplitude', 'qrs_width_ms', 'qrs_area', 'qrs_max', 'qrs_min',
                       'qrs_range', 'qrs_skew', 'qrs_kurt'}
norm_features = {'r_amplitude_norm', 'qrs_width_ms_norm', 'qrs_area_norm'}
pca_features = {f'pca_{i}' for i in range(10)}

categories = {}
for feat in FEATURE_COLS:
    if feat in rr_features:
        categories[feat] = 'RR-based'
    elif feat in norm_features:
        categories[feat] = 'Per-patient norm'
    elif feat in pca_features:
        categories[feat] = 'PCA'
    elif feat in morphology_features:
        categories[feat] = 'Raw morphology'
    else:
        categories[feat] = 'Other'

# Average shift by category
df_shift['Category'] = df_shift['Feature'].map(categories)
print("\n=== Average feature shift by category ===")
for lead_name in ['II', 'I', 'V1', 'V2']:
    sub = df_shift[df_shift['Lead'] == lead_name]
    cat_shift = sub.groupby('Category')['Shift (|d|)'].mean().sort_values(ascending=False)
    print(f"\nLead {lead_name}:")
    print(cat_shift.to_string(float_format='{:.3f}'.format))

## Retrain on INCART — upper bound comparison

Train a fresh GB on INCART Lead II data (50/50 record split) to establish the **upper bound** for INCART classification. This separates "lead mismatch" from "dataset difficulty" — if the retrained model also struggles, INCART is inherently harder (different patient population, annotation style, etc.).

In [ ]:
# Split INCART records 50/50 for train/test (by record, not by beat)
df_incart_ii = incart_dfs['II']
incart_records = sorted(df_incart_ii['record'].unique())
n_split = len(incart_records) // 2
incart_train_recs = set(incart_records[:n_split])
incart_test_recs = set(incart_records[n_split:])

incart_train = df_incart_ii[df_incart_ii['record'].isin(incart_train_recs)].copy()
incart_test = df_incart_ii[df_incart_ii['record'].isin(incart_test_recs)].copy()

print(f"INCART train: {len(incart_train)} beats from {len(incart_train_recs)} records")
print(f"INCART test:  {len(incart_test)} beats from {len(incart_test_recs)} records")
print(f"\nTrain classes: {incart_train['clinical_label'].value_counts().to_dict()}")
print(f"Test classes:  {incart_test['clinical_label'].value_counts().to_dict()}")

# Train GB on INCART Lead II
X_incart_train = incart_train[FEATURE_COLS].values
y_incart_train = incart_train['clinical_label'].values
sw_incart = compute_sample_weight('balanced', y_incart_train)

clf_incart = HistGradientBoostingClassifier(
    max_iter=300, max_depth=6, learning_rate=0.1,
    min_samples_leaf=20, l2_regularization=1.0, random_state=42
)
clf_incart.fit(X_incart_train, y_incart_train, sample_weight=sw_incart)

# Evaluate on INCART test
X_incart_test = incart_test[FEATURE_COLS].values
y_incart_test = incart_test['clinical_label'].values
y_pred_incart = clf_incart.predict(X_incart_test)

print("\n=== INCART Lead II: Retrained GB (upper bound) ===")
print(classification_report(y_incart_test, y_pred_incart, digits=3))
retrained_macro_f1 = f1_score(y_incart_test, y_pred_incart, average='macro', zero_division=0)
print(f"Macro F1: {retrained_macro_f1:.3f}")

# Also evaluate MIT-BIH model on same test split for direct comparison
y_pred_transfer = clf.predict(X_incart_test)
transfer_macro_f1 = f1_score(y_incart_test, y_pred_transfer, average='macro', zero_division=0)

print(f"\n=== Same test set comparison ===")
print(f"MIT-BIH-trained model (transfer):  Macro F1 = {transfer_macro_f1:.3f}")
print(f"INCART-retrained model (native):   Macro F1 = {retrained_macro_f1:.3f}")
print(f"Gap (transfer penalty):            {retrained_macro_f1 - transfer_macro_f1:+.3f}")

## Visualization — cross-lead comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Macro F1 by lead and model type
ax = axes[0]
leads_order = ['MIT-BIH MLII', 'II', 'I', 'V1', 'V2']
for model_name, marker, color in [('Full (34)', 'o', 'tab:blue'), ('RR-only (13)', 's', 'tab:orange')]:
    sub = df_results[df_results['Model'] == model_name]
    # Ensure correct ordering
    sub_ordered = sub.set_index('Lead').reindex(leads_order)
    ax.plot(range(len(leads_order)), sub_ordered['Macro F1'].values, marker=marker,
            label=model_name, color=color, linewidth=2, markersize=8)
ax.set_xticks(range(len(leads_order)))
ax.set_xticklabels(leads_order, rotation=15)
ax.set_ylabel('Macro F1')
ax.set_title('Macro F1 by Lead')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

# Plot 2: Per-class F1 for Full model across leads
ax = axes[1]
sub_full = df_results[df_results['Model'] == 'Full (34)'].set_index('Lead').reindex(leads_order)
for cls, color in [('N', 'tab:green'), ('V', 'tab:red'), ('S', 'tab:purple')]:
    ax.plot(range(len(leads_order)), sub_full[f'{cls} F1'].values, marker='o',
            label=f'{cls} F1', color=color, linewidth=2, markersize=8)
ax.set_xticks(range(len(leads_order)))
ax.set_xticklabels(leads_order, rotation=15)
ax.set_ylabel('F1 Score')
ax.set_title('Per-Class F1 (Full model)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

# Plot 3: Feature shift by category (Lead II vs Lead I)
ax = axes[2]
cat_order = ['PCA', 'Raw morphology', 'Per-patient norm', 'RR-based', 'Other']
x_pos = np.arange(len(cat_order))
width = 0.35
for i, (lead_name, color) in enumerate([('II', 'tab:blue'), ('I', 'tab:orange')]):
    sub = df_shift[df_shift['Lead'] == lead_name]
    cat_means = sub.groupby('Category')['Shift (|d|)'].mean()
    vals = [cat_means.get(c, 0) for c in cat_order]
    ax.bar(x_pos + i * width - width/2, vals, width, label=f'Lead {lead_name}', color=color, alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(cat_order, rotation=25, ha='right')
ax.set_ylabel('Mean |shift| (std units)')
ax.set_title('Feature Distribution Shift')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('cross_lead_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: cross_lead_validation.png")

## Confusion matrices — Lead II and Lead I

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
class_order = ['N', 'S', 'V']

configs = [
    ('MIT-BIH DS2 (baseline)', y_test, clf.predict(X_test)),
    ('INCART Lead II (transfer)', incart_dfs['II']['clinical_label'].values,
     clf.predict(incart_dfs['II'][FEATURE_COLS].values)),
    ('INCART Lead I (transfer)', incart_dfs['I']['clinical_label'].values,
     clf.predict(incart_dfs['I'][FEATURE_COLS].values)),
]

for ax, (title, y_true, y_pred) in zip(axes, configs):
    cm = confusion_matrix(y_true, y_pred, labels=class_order)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    
    im = ax.imshow(cm_pct, cmap='Blues', vmin=0, vmax=100)
    for i in range(3):
        for j in range(3):
            color = 'white' if cm_pct[i, j] > 50 else 'black'
            ax.text(j, i, f'{cm[i,j]}\n({cm_pct[i,j]:.1f}%)',
                    ha='center', va='center', fontsize=9, color=color)
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(class_order)
    ax.set_yticklabels(class_order)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    ax.set_title(f'{title}\nMacro F1: {macro_f1:.3f}')

plt.tight_layout()
plt.savefig('cross_lead_confusion.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: cross_lead_confusion.png")

## Summary and conclusions

### Cross-lead transfer results (MIT-BIH MLII → INCART)

| Source | Model | Accuracy | N F1 | S F1 | V F1 | Macro F1 |
|--------|-------|----------|------|------|------|----------|
| MIT-BIH MLII | Full (34) | 94.9% | 0.972 | 0.228 | 0.907 | **0.702** |
| MIT-BIH MLII | RR-only (13) | 86.5% | 0.931 | 0.129 | 0.544 | 0.534 |
| INCART Lead II | Full (34) | 92.8% | 0.961 | 0.157 | 0.860 | 0.660 |
| INCART Lead I | Full (34) | 92.4% | 0.969 | 0.269 | 0.744 | 0.661 |
| INCART Lead V1 | Full (34) | 91.5% | 0.954 | 0.129 | 0.849 | 0.644 |
| INCART Lead V2 | Full (34) | 91.0% | 0.956 | 0.155 | 0.815 | 0.642 |
| INCART all leads | RR-only (13) | 85.7% | 0.924 | 0.104 | 0.741 | 0.589 |
| INCART II retrained | Full (34) | 98.2% | 0.991 | 0.313 | 0.944 | **0.749** |

### Key findings

1. **Cross-lead transfer penalty is moderate**: Macro F1 drops from 0.702 → ~0.660 (-0.042) going MLII → INCART Lead II. Most of the loss is in V F1 (-0.047) and S F1 (-0.071); N F1 holds up well (-0.011).

2. **Full model beats RR-only on all INCART leads**: Despite morphology features shifting across leads, the full model (0.660 macro) still outperforms RR-only (0.589 macro) by +0.071. The morphology features add value even with lead mismatch.

3. **Lead I (wearable-relevant) performs comparably to Lead II**: Lead I macro F1 (0.661) matches Lead II (0.660). S F1 is actually *better* on Lead I (0.269 vs 0.157) — likely because Lead I P-wave morphology is more distinctive.

4. **Feature shift confirms expectations**: Raw morphology features shift most across leads (d=0.816 for Lead I vs d=0.153 for Lead II). RR-based features shift minimally (d=0.051). Per-patient normalization eliminates shift entirely (d=0.000).

5. **Retraining closes much of the gap**: INCART-retrained model achieves 0.749 macro F1 vs 0.668 for MIT-BIH transfer (same test set) — a +0.081 improvement. This quantifies the "dataset mismatch" penalty that retraining eliminates.

6. **RR-only features transfer perfectly across leads** (identical results for all 4 INCART leads, as expected — all RR features are lead-independent).

### Implications for wearable deployment

- **Initial deployment**: Use the full 34-feature model, not RR-only. Even with lead mismatch, morphology features still help.
- **Per-patient normalization works**: Zero distribution shift for normalized features confirms the calibration approach in the dashboard is sound.
- **Retraining on target lead data is worthwhile**: +0.081 macro F1 gain justifies collecting labeled data on the target hardware/lead configuration.
- **Lead I is viable for wearable**: No performance penalty vs Lead II, which is good news for wrist/chest-strap form factors that approximate Lead I.